Importing packages

In [1]:
from langchain.messages import SystemMessage, HumanMessage

from langchain.chat_models import BaseChatModel

In [2]:
from re import search

from typing import TypedDict

from enum import Enum

from json import load, dump

from os import listdir

In [3]:
from src.chat import get_chat

Configurations

In [4]:
DATA_DIR = "data"

TO_GENERATE = f"{DATA_DIR}/to-generate"
GENERATED = f"{DATA_DIR}/generated"

In [5]:
SYSTEM_SEMANTIC_MESSAGE = """Você é um assistente especializado em corrigir provas de assuntos variados, a sua função é receber um critério de aceitação da questão e fazer uma análise textual da resposta a partir deste critério.
Não faça nenhuma inferência da resposta que não seja a partir do critério da questão.
"""

In [6]:
SYSTEM_EXAMPLE_MESSAGE = """Você é um assistente especializado em corrigir provas de assuntos variados, a sua função é receber uma resposta de exemplo dada pelo professor e realizar uma comparação com a resposta dada pelo aluno.
Não faça nenhuma inferência da resposta do aluno que não seja proveniente da resposta exemplo."""

In [7]:
CRITERIA_SEMANTIC_MESSAGE = """Critério de aceitação da questão:"""

In [8]:
CRITERIA_EXAMPLE_MESSAGE = """Resposta de exemplo dada pelo professor:"""

Models

In [9]:
class CriteriaEnum(Enum):
    SEMANTIC = "SEMANTIC"
    EXAMPLE = "EXAMPLE"
    KEYWORD = "KEYWORD"
    
MAP_CRITERIA_ENUM = {
    "SEMANTIC": CriteriaEnum.SEMANTIC,
    "EXAMPLE": CriteriaEnum.EXAMPLE,
    "KEYWORD": CriteriaEnum.KEYWORD
}

In [10]:
class QuestionCriteria(TypedDict):
    id: int
    type: CriteriaEnum
    criteria: str
    
class AnswerStudent(TypedDict):
    id: int
    statement: str
    question_criteria: list[QuestionCriteria]
    answer: str

In [11]:
class Comment(TypedDict):
    criteria_id: int
    content: str

Messages

In [12]:
system_semantic_message = SystemMessage(content=SYSTEM_SEMANTIC_MESSAGE)
system_example_message = SystemMessage(content=SYSTEM_EXAMPLE_MESSAGE)

Correction Generator

In [13]:
def generate_keyword_comment(answer: str, criteria: QuestionCriteria) -> Comment:
    possible_match = search(criteria["criteria"], answer)
    if possible_match:
        possible_match_group = possible_match.group(0)
        content = f"A resposta contém a palavra chave esperada: '{possible_match_group}'."
    else:
        content = "A resposta não contém os elementos esperados conforme o critério de aceitação."
    return Comment(criteria_id=criteria["id"], content=content)

In [14]:
def generate_semantic_comment(chat: BaseChatModel, answer: str, criteria: QuestionCriteria) -> list[Comment]:
    human_criteria_message = HumanMessage(content=f"{CRITERIA_SEMANTIC_MESSAGE}: {criteria['criteria']}")
    human_answer_message = HumanMessage(content=f"Resposta do aluno: {answer}")
    
    response = chat.generate(
        messages=[
            [system_semantic_message, human_criteria_message, human_answer_message]
        ]
    )
    
    comments = [generation.message.content for generation in response.generations[0]]
    
    return [Comment(criteria_id=criteria["id"], content=comment) for comment in comments]

In [15]:
def generate_example_comment(chat: BaseChatModel, answer: str, criteria: QuestionCriteria) -> list[Comment]:
    human_criteria_message = HumanMessage(content=f"{CRITERIA_EXAMPLE_MESSAGE}: {criteria['criteria']}")
    human_answer_message = HumanMessage(content=f"Resposta do aluno: {answer}")
    
    response = chat.generate(
        messages=[
            [system_example_message, human_criteria_message, human_answer_message]
        ]
    )
    
    comments = [generation.message.content for generation in response.generations[0]]
    
    return [Comment(criteria_id=criteria["id"], content=comment) for comment in comments]

In [16]:
def generate_comments(question: AnswerStudent, chat: BaseChatModel) -> list[Comment]:
    criteria_keywords = [c for c in question["question_criteria"] if MAP_CRITERIA_ENUM[c["type"]] == CriteriaEnum.KEYWORD]
    criteria_semantic = [c for c in question["question_criteria"] if MAP_CRITERIA_ENUM[c["type"]] == CriteriaEnum.SEMANTIC]
    criteria_example = [c for c in question["question_criteria"] if MAP_CRITERIA_ENUM[c["type"]] == CriteriaEnum.EXAMPLE]
    
    comments = []
    
    coments_keyword: list[Comment] = list(map(lambda c: generate_keyword_comment(question["answer"], c), criteria_keywords))    
    coments_generate_example_semantic = [generate_semantic_comment(chat, question["answer"], c) for c in criteria_semantic]
    coments_generate_example_example = [generate_example_comment(chat, question["answer"], c) for c in criteria_example]
    
    comments.extend(coments_keyword)
    for comment_list in coments_generate_example_semantic:
        comments.extend(comment_list)
    for comment_list in coments_generate_example_example:
        comments.extend(comment_list)
    
    return comments

Generating Comments

In [17]:
path_to_generate = listdir(TO_GENERATE)

for filename in path_to_generate:
    with open(f"{TO_GENERATE}/{filename}", "r") as f:
        data: AnswerStudent = load(f)
        chat_model = get_chat()
        comments = generate_comments(data, chat_model)
        with open (f"{GENERATED}/{filename}", "w") as f_generated:
            dump({"comments": comments}, f_generated, indent=4, ensure_ascii=False)